# Kaggle Tuned Ensemble Submission

This notebook trains two TF-IDF Logistic Regression models with A/B swap augmentation, applies swap-test averaging, blends their probabilities with weights 0.7 and 0.3, and saves `/kaggle/working/submission.csv`.

## 1. Imports and Auto Path Search

Find Kaggle input files automatically under `/kaggle/input`. No local Windows paths are used.

In [ ]:
import ast
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
LABELS = [0, 1, 2]
SUBMISSION_COLUMNS = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
PROBABILITY_COLUMNS = ['winner_model_a', 'winner_model_b', 'winner_tie']

INPUT_ROOT = Path('/kaggle/input')
submission_path = Path('/kaggle/working/submission.csv')


train_candidates = list(INPUT_ROOT.rglob('train.csv'))
test_candidates = list(INPUT_ROOT.rglob('test.csv'))
sample_candidates = list(INPUT_ROOT.rglob('sample_submission.csv'))

print('train candidates:')
for path in train_candidates:
    print(path)

print('test candidates:')
for path in test_candidates:
    print(path)

print('sample_submission candidates:')
for path in sample_candidates:
    print(path)

if len(train_candidates) == 0:
    raise FileNotFoundError('train.csv not found under /kaggle/input')
if len(test_candidates) == 0:
    raise FileNotFoundError('test.csv not found under /kaggle/input')
if len(sample_candidates) == 0:
    raise FileNotFoundError('sample_submission.csv not found under /kaggle/input')

train_path = train_candidates[0]
test_path = test_candidates[0]
sample_submission_path = sample_candidates[0]

print('train_path:', train_path)
print('test_path:', test_path)
print('sample_submission_path:', sample_submission_path)
print('submission_path:', submission_path)

## 2. Read Kaggle Data

Read train, test, and sample submission files.

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample_submission shape:', sample_submission.shape)
print(f'train columns: {train.columns.tolist()}')
print(f'test columns: {test.columns.tolist()}')
print(f'sample_submission columns: {sample_submission.columns.tolist()}')

## 3. Text Cleaning and Feature Functions

Parse JSON/list-like text fields, clean invalid Unicode, build model input text, and create A/B swapped data.

In [ ]:
def clean_unicode(text):
    if text is None:
        return ''
    if not isinstance(text, str):
        text = str(text)

    try:
        text = text.encode('utf-16', 'surrogatepass').decode('utf-16', 'replace')
    except Exception:
        text = text.encode('utf-8', 'ignore').decode('utf-8', 'ignore')

    text = ''.join(ch for ch in text if not (0xD800 <= ord(ch) <= 0xDFFF))
    return text


def parse_text_field(x):
    if not isinstance(x, str):
        x = str(x)

    text = x.strip()

    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', SyntaxWarning)
            return ast.literal_eval(text)
    except (ValueError, SyntaxError, TypeError):
        return text


def normalize_text(x):
    if x is None:
        return ''

    try:
        if pd.isna(x):
            return ''
    except (TypeError, ValueError):
        pass

    if not isinstance(x, str):
        x = str(x)

    parsed = parse_text_field(x.strip())
    if isinstance(parsed, list):
        text = '\n'.join(str(item) for item in parsed if item is not None)
    else:
        text = str(parsed)

    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace(chr(0x2028), '\n').replace(chr(0x2029), '\n').strip()
    text = clean_unicode(text)
    return text


def make_text_input(df):
    prompt = df['prompt_clean'].fillna('').astype(str)
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    return (
        'Prompt:\n' + prompt
        + '\n\nResponse A:\n' + response_a
        + '\n\nResponse B:\n' + response_b
    )


def swap_ab_dataframe(df):
    swapped = df.copy()
    swapped['response_a_clean'] = df['response_b_clean'].values
    swapped['response_b_clean'] = df['response_a_clean'].values

    if 'label' in swapped.columns:
        swapped['label'] = df['label'].map({0: 1, 1: 0, 2: 2}).astype(int).values

    return swapped


def align_probabilities(probabilities, classes, labels=LABELS):
    aligned = np.zeros((probabilities.shape[0], len(labels)), dtype=float)
    class_to_position = {int(label): idx for idx, label in enumerate(classes)}
    for output_position, label in enumerate(labels):
        if label in class_to_position:
            aligned[:, output_position] = probabilities[:, class_to_position[label]]
    return aligned


def transform_word_char(word_vectorizer, char_vectorizer, text_series):
    word_features = word_vectorizer.transform(text_series)
    char_features = char_vectorizer.transform(text_series)
    return hstack([word_features, char_features], format='csr')


print('Functions ready.')

## 4. Clean Text and Convert Labels

Create clean text fields for train and test, then convert train labels to 0, 1, and 2.

In [ ]:
train_raw_columns = {
    'id',
    'prompt',
    'response_a',
    'response_b',
    'winner_model_a',
    'winner_model_b',
    'winner_tie',
}
test_raw_columns = {'id', 'prompt', 'response_a', 'response_b'}

missing_train_raw_columns = sorted(train_raw_columns - set(train.columns))
missing_test_raw_columns = sorted(test_raw_columns - set(test.columns))

if missing_train_raw_columns:
    raise ValueError(f'train.csv missing columns: {missing_train_raw_columns}')
if missing_test_raw_columns:
    raise ValueError(f'test.csv missing columns: {missing_test_raw_columns}')

for df_name, df in [('train', train), ('test', test)]:
    for column in ['prompt', 'response_a', 'response_b']:
        df[f'{column}_clean'] = df[column].apply(normalize_text)
    df['text_input'] = make_text_input(df)
    print(f'{df_name}: clean fields and text_input created.')

train['label'] = np.select(
    [
        train['winner_model_a'] == 1,
        train['winner_model_b'] == 1,
        train['winner_tie'] == 1,
    ],
    [0, 1, 2],
    default=-1,
).astype(int)

if (train['label'] == -1).any():
    raise ValueError('Invalid label rows found')

print('Label counts:')
print(train['label'].value_counts().sort_index())

## 5. A/B Swap Augmentation

Augment the full training data by swapping response A and response B, then flipping A-win and B-win labels.

In [ ]:
train_original = train.copy()
swapped_train = swap_ab_dataframe(train_original)
train_aug = pd.concat([train_original, swapped_train], ignore_index=True)
train_aug = train_aug.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

train_aug['text_input'] = make_text_input(train_aug)
test['text_input'] = make_text_input(test)

print(f'Original train size: {len(train_original)}')
print(f'Swapped train size: {len(swapped_train)}')
print(f'train_aug shape: {train_aug.shape}')
print('Original label counts:')
print(train_original['label'].value_counts().sort_index())
print('Swapped label counts:')
print(swapped_train['label'].value_counts().sort_index())
print('Augmented label counts:')
print(train_aug['label'].value_counts().sort_index())

## 6. Model 1: Word-Level TF-IDF A/B Swap Model

Train word-level TF-IDF Logistic Regression with C=0.1 on the full augmented training data.

In [ ]:
word_vectorizer_1 = TfidfVectorizer(
    analyzer='word',
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)

X_train_word_1 = word_vectorizer_1.fit_transform(train_aug['text_input'])
y_train = train_aug['label'].astype(int)

model_1 = LogisticRegression(
    C=0.1,
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
model_1.fit(X_train_word_1, y_train)

print(f'Model 1 feature matrix shape: {X_train_word_1.shape}')
print(f'Model 1 classes: {model_1.classes_.tolist()}')

## 7. Predict Model 1 with Swap-Test Averaging

Predict original and swapped test inputs, then map swapped probabilities back to the original A/B order.

In [ ]:
def predict_with_swap_average_word_model(model, vectorizer, test_df):
    original_text = make_text_input(test_df)
    X_original = vectorizer.transform(original_text)
    probs_original = align_probabilities(model.predict_proba(X_original), model.classes_, LABELS)

    test_swapped = swap_ab_dataframe(test_df.copy())
    swapped_text = make_text_input(test_swapped)
    X_swapped = vectorizer.transform(swapped_text)
    probs_swapped = align_probabilities(model.predict_proba(X_swapped), model.classes_, LABELS)

    mapped_probs_swapped = np.column_stack([
        probs_swapped[:, 1],
        probs_swapped[:, 0],
        probs_swapped[:, 2],
    ])
    avg_probs = 0.5 * probs_original + 0.5 * mapped_probs_swapped
    return avg_probs


probs_ab_swap = predict_with_swap_average_word_model(model_1, word_vectorizer_1, test)
print(f'probs_ab_swap shape: {probs_ab_swap.shape}')

del X_train_word_1
gc.collect()

## 8. Model 2: Tuned Word+Char TF-IDF A/B Swap Model

Train word+char TF-IDF Logistic Regression with tuned C=0.05 on the same augmented training data.

In [ ]:
word_vectorizer_2 = TfidfVectorizer(
    analyzer='word',
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)

char_vectorizer_2 = TfidfVectorizer(
    analyzer='char_wb',
    max_features=50000,
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_word_2 = word_vectorizer_2.fit_transform(train_aug['text_input'])
X_train_char_2 = char_vectorizer_2.fit_transform(train_aug['text_input'])
X_train_word_char_2 = hstack([X_train_word_2, X_train_char_2], format='csr')

model_2 = LogisticRegression(
    C=0.05,
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
model_2.fit(X_train_word_char_2, y_train)

print(f'Model 2 word feature matrix shape: {X_train_word_2.shape}')
print(f'Model 2 char feature matrix shape: {X_train_char_2.shape}')
print(f'Model 2 combined feature matrix shape: {X_train_word_char_2.shape}')
print(f'Model 2 classes: {model_2.classes_.tolist()}')

## 9. Predict Model 2 with Swap-Test Averaging

Use word+char features for original and swapped test inputs, then average mapped probabilities.

In [ ]:
def predict_with_swap_average_word_char_model(model, word_vectorizer, char_vectorizer, test_df):
    original_text = make_text_input(test_df)
    X_original = transform_word_char(word_vectorizer, char_vectorizer, original_text)
    probs_original = align_probabilities(model.predict_proba(X_original), model.classes_, LABELS)

    test_swapped = swap_ab_dataframe(test_df.copy())
    swapped_text = make_text_input(test_swapped)
    X_swapped = transform_word_char(word_vectorizer, char_vectorizer, swapped_text)
    probs_swapped = align_probabilities(model.predict_proba(X_swapped), model.classes_, LABELS)

    mapped_probs_swapped = np.column_stack([
        probs_swapped[:, 1],
        probs_swapped[:, 0],
        probs_swapped[:, 2],
    ])
    avg_probs = 0.5 * probs_original + 0.5 * mapped_probs_swapped
    return avg_probs


probs_tuned_word_char = predict_with_swap_average_word_char_model(
    model_2,
    word_vectorizer_2,
    char_vectorizer_2,
    test,
)
print(f'probs_tuned_word_char shape: {probs_tuned_word_char.shape}')

## 10. Probability Ensemble

Blend model 1 and model 2 probabilities with weights 0.7 and 0.3.

In [ ]:
final_probs = 0.7 * probs_ab_swap + 0.3 * probs_tuned_word_char

print(f'final_probs shape: {final_probs.shape}')
print(f'final probability sums close to 1: {np.allclose(final_probs.sum(axis=1), 1.0, atol=1e-6)}')

if not np.allclose(final_probs.sum(axis=1), 1.0, atol=1e-6):
    raise ValueError('Final probability rows do not sum to 1.')

## 11. Create and Validate Submission

Create Kaggle submission and run final checks.

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

probability_sum = submission[PROBABILITY_COLUMNS].sum(axis=1)

print(f'submission shape: {submission.shape}')
print(f'submission columns: {submission.columns.tolist()}')
print(f'submission columns correct: {submission.columns.tolist() == SUBMISSION_COLUMNS}')
print(f'probability sums close to 1: {np.allclose(probability_sum, 1.0, atol=1e-6)}')
print(f'submission has NaN: {submission.isna().any().any()}')
print('submission head:')
print(submission.head())

if submission.columns.tolist() != SUBMISSION_COLUMNS:
    raise ValueError('Submission columns are not correct.')
if not np.allclose(probability_sum, 1.0, atol=1e-6):
    raise ValueError('Submission probability rows do not sum to 1.')
if submission.isna().any().any():
    raise ValueError('Submission contains NaN values.')

## 12. Save Submission

Save final output to `/kaggle/working/submission.csv`.

In [ ]:
submission.to_csv(submission_path, index=False)

print(f'Saved submission: {submission_path}')
print('Kaggle tuned ensemble submission finished successfully.')